# E2b — Leaky vs clean on RT-DETR-L (cross-model check for E2)
**Purpose:** show that leakage-driven mAP inflation is NOT model-dependent. E2 measured it on YOLO26s (CNN); E2b repeats the identical protocol on RT-DETR-L (Transformer). If both models show comparable inflation, the paper's central claim is twice as solid.

- **leaky**: the authors' ORIGINAL split (augmentation before splitting)
- **clean**: pool all images and re-split 70/20/10 by source-image GROUP (group-aware, seed 42)
- Same epochs (15), same seed, same imgsz 640 as E2 — only the model changes.

**How to run:** GPU **T4 x2** · Internet **ON** · set `ROBOFLOW_API_KEY` (a NEW key) · Save & Run All.
No Add Input required. Estimated **~7-8 h** (RT-DETR is ~2.5× heavier than YOLO; QUICK_TEST=True: ~40 min).
**Two-level checkpoint-resume:** per training run (state.json) AND mid-run (last.pt).
If the 12 h limit cuts the session: Save Version → Edit → Add Input → select this notebook's output → Run All again.

In [ ]:
# ==================== CONFIG (E2b) ====================
QUICK_TEST = False
EPOCHS   = 2 if QUICK_TEST else 15
FRACTION = 0.15 if QUICK_TEST else 1.0
SEED = 42
ROBOFLOW_API_KEY = ""   # <== PUT YOUR NEW KEY HERE (never make the notebook public while a key is in it)

import os, random, shutil, glob, json, time, yaml
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)

STATE_PATH = f"{WORK}/state.json"
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)

# RESUME level 1: load state.json from E2b's own previous output
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any("_rtdetr_" in k for k in _s):                      # matches the E2b signature
        # GUARD: reject state from a QUICK_TEST run (every final run under 5 minutes)
        mins = [v.get("min", 0) for v in _s.values() if isinstance(v, dict) and "min" in v]
        if not QUICK_TEST and mins and max(mins) < 5:
            print("SKIPPING quick-test state:", _st); continue
        STATE = _s; shutil.copy(_st, STATE_PATH)
        print("E2b CHECKPOINT LOADED FROM:", _st); break
if not STATE and os.path.exists(STATE_PATH):
    STATE = json.load(open(STATE_PATH))
def done(k): return k in STATE
def mark(k, v): STATE[k] = v; save_state()
assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY (a NEW key — any previously exposed key must be revoked)!"
print("[E2b] QUICK_TEST=", QUICK_TEST, "| runs done:", [k for k in STATE if "_rtdetr_" in k] or "(none yet)")

In [ ]:
%pip install -q ultralytics roboflow
import torch
assert torch.cuda.is_available(), "Enable GPU T4!"
cap = torch.cuda.get_device_capability(0)
assert cap[0] >= 7, f"GPU sm_{cap[0]}{cap[1]} is too old — select T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")

## 1. Download the two source datasets (keeping the authors' leaky split)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
DATASETS = {
    "B_testcasque": ("testcasque", "ppe-detection-qlq3d", 1),
    "C_mohamed":    ("mohamed-traore-2ekkp", "ppe-detection-l80fg", 5),
}
DS_DIR = {}
for name, (ws, proj, ver) in DATASETS.items():
    out = f"{WORK}/{name}"
    if not os.path.exists(f"{out}/data.yaml"):
        rf.workspace(ws).project(proj).version(ver).download("yolov8", location=out)
    DS_DIR[name] = out
    n = sum(len(glob.glob(f"{out}/{sp}/images/*")) for sp in ["train","valid","test"])
    print(f"{name}: {n} images ->", out)

## 2. Build the CLEAN variant — re-split 70/20/10 by source-image group (identical to E2)

In [ ]:
def src_of(path):
    s = os.path.basename(path)
    return s.split(".rf.")[0] if ".rf." in s else os.path.splitext(s)[0]

def fix_yaml(root):
    y0 = yaml.safe_load(open(f"{root}/data.yaml"))
    names = y0["names"]
    y = {"path": root, "train": "train/images", "val": "valid/images", "test": "test/images",
         "names": names, "nc": len(names)}
    p = f"{root}/data_fixed.yaml"; yaml.safe_dump(y, open(p, "w")); return p

CLEAN_DIR, LEAKY_YAML, CLEAN_YAML = {}, {}, {}
for name, root in DS_DIR.items():
    items = []
    for sp in ["train", "valid", "test"]:
        for img in sorted(glob.glob(f"{root}/{sp}/images/*")):
            lbl = f"{root}/{sp}/labels/{os.path.splitext(os.path.basename(img))[0]}.txt"
            items.append((img, lbl if os.path.exists(lbl) else None, sp))
    groups = {}
    for it in items: groups.setdefault(src_of(it[0]), []).append(it)
    out_root = f"{WORK}/{name}_clean"
    if not os.path.exists(f"{out_root}/data.yaml"):
        for sp in ["train", "valid", "test"]:
            os.makedirs(f"{out_root}/{sp}/images", exist_ok=True)
            os.makedirs(f"{out_root}/{sp}/labels", exist_ok=True)
        keys = sorted(groups); random.Random(SEED).shuffle(keys)
        total = len(items); acc = 0
        for k in keys:
            sp = "train" if acc < 0.7 * total else ("valid" if acc < 0.9 * total else "test")
            for img, lbl, _ in groups[k]:
                shutil.copy(img, f"{out_root}/{sp}/images/")
                if lbl: shutil.copy(lbl, f"{out_root}/{sp}/labels/")
                acc += 1
        shutil.copy(f"{root}/data.yaml", f"{out_root}/data.yaml")
    CLEAN_DIR[name] = out_root
    LEAKY_YAML[name] = fix_yaml(root); CLEAN_YAML[name] = fix_yaml(out_root)
    for sp in ["train", "valid", "test"]:
        print(f"  {name}_clean/{sp}:", len(glob.glob(f"{out_root}/{sp}/images/*")), "images")

## 3. Train RT-DETR-L: 4 runs (B-leaky, B-clean, C-leaky, C-clean) — AdamW lr=1e-4 (from the E1 HPO), mid-run resume via last.pt

In [ ]:
from ultralytics import RTDETR
import pandas as pd, gc
rows = []
for name in DS_DIR:
    for arm, dyaml in [("leaky", LEAKY_YAML[name]), ("clean", CLEAN_YAML[name])]:
        key = f"{name}_rtdetr_{arm}"
        if done(key):
            rows.append(STATE[key]); print("skip:", STATE[key]); continue
        t0 = time.time()
        last = f"{WORK}/runs/{key}/weights/last.pt"
        if os.path.exists(last):
            print("MID-RUN RESUME from:", last)
            m = RTDETR(last)
            m.train(resume=True)
        else:
            m = RTDETR("rtdetr-l.pt")
            m.train(data=dyaml, epochs=EPOCHS, imgsz=640, batch=8, seed=SEED, fraction=FRACTION,
                    optimizer="AdamW", lr0=1e-4,
                    project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
        v = m.val(data=dyaml, split="val", verbose=False)
        t = m.val(data=dyaml, split="test", verbose=False)
        row = {"dataset": name, "model": "RT-DETR-L", "arm": arm, "epochs": EPOCHS,
               "mAP50_val": round(v.box.map50, 4), "mAP50_test": round(t.box.map50, 4),
               "mAP50_95_test": round(t.box.map, 4), "min": round((time.time() - t0) / 60, 1)}
        rows.append(row); mark(key, row); print(row)
        del m; gc.collect(); torch.cuda.empty_cache()

## 4. Results: inflation per model (compared against YOLO26s from E2)

In [ ]:
df = pd.DataFrame(rows)
piv = df.pivot(index="dataset", columns="arm", values="mAP50_test")
piv["inflation_mAP50"] = (piv["leaky"] - piv["clean"]).round(4)
piv95 = df.pivot(index="dataset", columns="arm", values="mAP50_95_test")
piv["inflation_mAP50_95"] = (piv95["leaky"] - piv95["clean"]).round(4)
display(df); display(piv)
# Cross-reference with E2 (YOLO26s) — the figures used in the paper
E2_YOLO = {"B_testcasque": 0.0192, "C_mohamed": 0.0406}
for ds, infl in E2_YOLO.items():
    if ds in piv.index:
        print(f"{ds}: inflation RT-DETR-L = {piv.loc[ds,'inflation_mAP50']:.4f} | YOLO26s (E2) = {infl:.4f}")
df.to_csv(f"{WORK}/results/e2b_runs.csv", index=False)
piv.to_csv(f"{WORK}/results/e2b_inflation.csv")
print("\n>>> E2b CONCLUSION: if both models show inflation of the same sign and magnitude, leakage is a property of the DATA, not of the model.")

In [ ]:
# Clean up: DELETE every downloaded dataset — the output keeps only results/ + runs/ + state.json
for name in list(DS_DIR):
    shutil.rmtree(DS_DIR[name], ignore_errors=True)
    shutil.rmtree(f"{WORK}/{name}_clean", ignore_errors=True)
# runs/ keeps the weights for resuming; once ALL FOUR runs are done, uncomment the two lines below for a lean output:
# if all(done(f"{n}_rtdetr_{a}") for n in DATASETS for a in ["leaky","clean"]):
#     shutil.rmtree(f"{WORK}/runs", ignore_errors=True)
print("Cleaned. Output = results/ + runs/ + state.json")